In [ ]:
import json
import random

path = '/mnt/shared-storage-gpfs2/sfteval/lsz/spid/verl-agent/checkpoints/textworld_express_grpo_all_qwen2.5-1.5b-instruct_sp_0.2_id_0.2_20260108_153658/step_60/iter_1/trajectories_step_60.jsonl'
data = [json.loads(line) for line in open(path)]
len(data)

In [ ]:
data[0].keys()

In [ ]:
data[0]

In [ ]:
item = data[0]
key_l = ['input','output']
for k in key_l:
    print(item[k])
    print('-'*100)

In [23]:
# data[0]['raw_data'].keys()
from collections import deque

def bfs(obj, parent_key=''):
    queue = deque([(obj, parent_key)])
    while queue:
        cur, cur_key = queue.popleft()
        if 'prob' in cur_key or 'advantage' in cur_key or 'position' in cur_key or 'input_ids' in cur_key or 'attention_mask' in cur_key or 'responses' in cur_key:
            continue
        if isinstance(cur, dict):
            for k, v in cur.items():
                # 拼接所有key前缀
                full_key = f"{cur_key}.{k}" if cur_key else k
                queue.append((v, full_key))
        elif isinstance(cur, list):
            # 直接打印整个list，不逐个打印
            if 'raw_prompt' in cur_key:
                print(f'【{cur_key}】')
                print(cur[0]['content'])
                print(cur[1]['content'])
            print(f"【{cur_key}】: {cur}")
        else:
            print(f"【{cur_key}】: {cur}")
item = random.choice(lines)
bfs(item)

【step_number】: 27
【history_len】: 2
【current_obs】: You take the used tissue.
【action】: take copybook
【next_obs】: You take the copybook.
【admissibles_count】: 14
【sp_input】: <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
You are an expert agent operating in an interactive environment.
Prior to this step, you have already taken 26 step(s). Below are the history observations and the corresponding actions you took:
[Observation 25: "You are in the pantry. In one part of the room you see a folding chair, that has nothing on it. There is also a shelf, that has nothing on it. 
To the North you see the kitchen. ", Action 25: "move north"]
[Observation 26: "You are in the kitchen. In one part of the room you see a stove. There is also an oven. You also see a fridge that is closed. In another part of the room you see a counter, that has nothing on it. In one part of the room you see an open kitchen cupboard, that is empty. There i

IndexError: list index out of range

In [25]:
path = '/mnt/shared-storage-gpfs2/sfteval/lsz/spid/verl-agent/checkpoints/textworld_express_grpo_all_qwen2.5-1.5b-instruct_sp_0.2_id_0.2_20260108_153658/aux_debug_rank0.jsonl'
prompts = []
from tqdm import tqdm
with open(path) as f:
    for line in tqdm(f):
        item = json.loads(line)
        prompt = item['raw_data']['raw_prompt']
        prompts.append(prompt)
len(prompts)
prompts[0]


133192it [03:11, 697.24it/s]


[{'content': "\nYou are an expert agent playing a text-based adventure game called coin.\nYour task is: Your task is to search the environment and find the coin.  Once you find the coin, take it.\nPrior to this step, you have already taken 1 step(s). Below are the most recent 1 observations and the corresponding actions you took: [Observation 1: 'You are in the kitchen. In one part of the room you see a stove. There is also an oven. You also see a fridge that is closed. In another part of the room you see a counter, that has nothing on it. In one part of the room you see a kitchen cupboard that is closed. There is also a cutlery drawer that is closed. You also see a trash can that is closed. In another part of the room you see a dishwasher that is closed. In one part of the room you see a dining chair, that has nothing on it. \nTo the North you see a closed plain door. To the South you see the corridor. To the East you see the living room. ', Action 1: 'move north']\nYou are now at ste

In [42]:
item = random.choice(prompts)
import re
content = item[0]['content']
def get_task(content):
    match = re.search(r'You are an expert agent playing a text-based adventure game called (.*?)\.', content)
    if match:
        return match.group(1).strip()
    else:
        return None
tasks = [get_task(item[0]['content']) for item in tqdm(prompts)]
from collections import Counter
Counter(tasks)

100%|██████████| 133192/133192 [00:00<00:00, 467520.90it/s]


Counter({'twc': 24103,
         'cookingworld': 22992,
         'mapreader': 21536,
         'coin': 20217,
         'arithmetic': 18313,
         'sorting': 14832,
         'peckingorder': 7046,
         'simonsays': 4153})

In [ ]:
item = random.choice(data)
item

# trajectory
# print(item['input'])
# item = data[110]

# spid train data

# print(item['messages'][0]['content'])
# print('-'*100)
# print(item['messages'][1]['content'])

In [ ]:
import json
path = '/mnt/shared-storage-gpfs2/sfteval/lsz/spid/verl-agent/data/alfworld/seq2seq_data/tw_alfred_seq2seq_train_task1_hc.json'
data = json.load(open(path))
print(len(data), len(data[0]))
data[0]
# print(data[0][1]['prompt'])

In [ ]:
from genson import SchemaBuilder
import json

builder = SchemaBuilder()
builder.add_object(data)
schema = builder.to_schema()
schema


In [ ]:
import random
item = random.choice(random.choice(data))
print(item['prompt'])
print(item['admissible_actions'])

In [ ]:
item = data[-1]
obs = item['obs']
actions = item['actions']
admissibles = item['admissibles']

i = 0
obs[i], actions[i], admissibles[i]



In [ ]:
from collections import defaultdict

# Group the data by uid
grouped_data = defaultdict(list)
for entry in data:
    grouped_data[entry['uid']].append(entry)

grouped_data


In [ ]:
for k,v in grouped_data.items():
    print(k)
    display(v)
    break

In [ ]:
k,v = list(grouped_data.items())[0]
def get_triplet(item):
    return item['last_obs'], item['last_action'], item['current_obs']
triplets = [get_triplet(item) for item in v]
for item in triplets:
    print(item)
    print('-'*100)


In [ ]:
list(grouped_data.items())[0][1][-2]

In [ ]:
from verl import DataProto
from transformers import AutoTokenizer

model_name = '/home/test/test06/wzt/models/qwen2.5-1.5b-instruct'
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
data_path = '/home/test/test06/wzt/code/verl-agent/checkpoints/collect_train/code_verl-agent_checkpoints_alfworld_grpo_his2_models_qwen2.5-1.5b-instruct_20251203_114254_global_step_30_actor_huggingface_20251208_001235/merged_trajectories.pkl'
data = DataProto.load_from_disk(data_path)

In [ ]:
data.non_tensor_batch.keys()

In [ ]:
# data.non_tensor_batch['rewards'].shape
# 先根据uid对数据进行分组，每组包含数据的下标区间（起始下标和结束下标）
from collections import defaultdict

uid_to_indices = defaultdict(list)
uids = data.non_tensor_batch['uid']
for idx, uid in enumerate(uids):
    uid_to_indices[uid].append(idx)

# 对每组的下标求区间（起止下标、包含起止均含）
uid_index_ranges = {}
for uid, indices in uid_to_indices.items():
    start = indices[0]
    end = indices[-1]
    uid_index_ranges[uid] = (start, end)

# 打印部分结果查看
# for i, (uid, (start, end)) in enumerate(uid_index_ranges.items()):
#     print(f"uid: {uid}, indices: [{start}, {end}]")
#     if i >= 4:
#         break

# Find UIDs where episode_rewards is 10.0
uids_with_max_rewards = [uid for uid, (start, end) in uid_index_ranges.items()
                         if any(data.non_tensor_batch['episode_rewards'][i] == 10.0 for i in range(start, end+1))]

print("UIDs with episode_rewards of 10.0:", uids_with_max_rewards)

group_idx = uids_with_max_rewards[0]
l = uid_index_ranges[group_idx][0]
r = uid_index_ranges[group_idx][1]
# for i in range(l,r+1):
#     print('-'*100)
#     print(data.non_tensor_batch['uid'][i])
#     print(data.non_tensor_batch['is_action_valid'][i])
#     print(data.non_tensor_batch['rewards'][i])
#     print(data.non_tensor_batch['episode_rewards'][i])
#     print(tokenizer.decode(data.batch['input_ids'][i],skip_special_tokens=True))
    

for group_idx in uids_with_max_rewards:
    l = uid_index_ranges[group_idx][0]
    r = uid_index_ranges[group_idx][1]
    for i in range(l,r+1):
        text = tokenizer.decode(data.batch['input_ids'][i],skip_special_tokens=True)
        
    

In [ ]:
print(tokenizer.decode(data.batch['input_ids'][0],skip_special_tokens=True))

In [ ]:
uids = data.non_tensor_batch['uid']
uids = set(uids)
len(uids)

In [ ]:
# prev_uid = data.non_tensor_batch['uid'][0]
# result = []
# for i in range(len(data.batch['input_ids'])):
#     uid = data.non_tensor_batch['uid'][i]
#     input_ids = data.batch['input_ids'][i]
#     text = tokenizer.decode(input_ids, skip_special_tokens=True)
#     if uid!=prev_uid:
#         break
#     result.append(text)
    
# 把每个uid对应的第一个input_id保存下来
first_input_ids_per_uid = {}
for i in range(len(data.batch['input_ids'])):
    uid = data.non_tensor_batch['uid'][i]
    if uid not in first_input_ids_per_uid:
        first_input_ids_per_uid[uid] = tokenizer.decode(data.batch['input_ids'][i], skip_special_tokens=True)
print(list(first_input_ids_per_uid.items())[0][1])


In [ ]:
len(set(first_input_ids_per_uid.values()))